In [0]:
sales_path='development_042_silver_sandbox.demand_forecast.sales_silver'
like_games_path='development_042_silver_sandbox.demand_forecast.clustering_core3_like_games'
time_series_path='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'
run_id='781ee4f234bf44089c79d7885d8462f9'#'4e21f8afc8db4b179167faebe7fbd0d5'
sales_cl='expr1_sum'
sales_cl_cumsum='cumsum_expr1_sum'
performance=['avg_coin_in_index_vs_house','avg_theo_net_win_index_vs_house']
weight_col=['no_of_slots']
result_dest='development_042_silver_sandbox.demand_forecast.time_series_extrapolated'

In [0]:
sales_df = spark.table(sales_path)
time_series_df = spark.table(time_series_path).filter("own_status = 'owned'")
like_games_df = spark.table(like_games_path)
like_games_df = like_games_df[like_games_df.mlflow_run_id == run_id]
display(like_games_df)
display(sales_df)
display(time_series_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Configuration ---
weight_col_name = weight_col[0]        # 'no_of_slots'
metric_cols = performance + [sales_cl]  # performance indices + sales
curve_length_method = 'min'             # how to determine curve length: 'min', 'max', 'avg'
max_curve_length = 12

# --- Step 1: Build per-like-game monthly time series ---
# Get mapping: test_game_name -> game_name (each test game has ~3 like games)
mapping_df = like_games_df.select("test_game_name", "game_name", "neighbor_rank").dropDuplicates(["test_game_name", "game_name"])

# Get performance time series for like games (game_name, yearmonth, performance, weight)
perf_df = time_series_df.select(
    F.col("game_name"),
    F.col("yearmonth"),
    F.col("own_status"),
    F.col(weight_col_name).cast("double").alias(weight_col_name),
    *[F.col(c).cast("double").alias(c) for c in performance]
)

# Get sales for like games (ep_theme_name, beginning_month_date, expr1_sum)
# Deduplicate sales to one record per (game, month, revenue_type)
unique_sales = sales_df.dropDuplicates(["beginning_month_date", "revenue_type", sales_cl, "ep_theme_name"])
sales_agg = unique_sales.groupBy("ep_theme_name", "beginning_month_date").agg(
    F.sum(F.col(sales_cl).cast("double")).alias(sales_cl)
)

# --- Step 2: Join performance and sales for each like game on date ---
# Outer join so performance months without sales get 0 and we keep full curves
game_ts = perf_df.join(
    sales_agg,
    (perf_df.game_name == sales_agg.ep_theme_name) & (perf_df.yearmonth == sales_agg.beginning_month_date),
    how="left"
).select(
    perf_df.game_name,
    perf_df.yearmonth,
    F.col("own_status"),
    F.col(weight_col_name),
    *[F.col(c) for c in performance],
    F.coalesce(F.col(sales_cl), F.lit(0.0)).alias(sales_cl)
)

# --- Step 3: Map to test games ---
train_data = mapping_df.join(
    game_ts,
    mapping_df.game_name == game_ts.game_name,
    "inner"
).select(
    "test_game_name",
    mapping_df.game_name,
    "neighbor_rank",
    "yearmonth",
    "own_status",
    weight_col_name,
    *[F.col(c) for c in performance],
    F.col(sales_cl)
)

# --- Step 4: Assign ordinal month index (removes calendar alignment) ---
w = Window.partitionBy("test_game_name", "game_name").orderBy("yearmonth")
train_data = train_data.withColumn("month_index", F.row_number().over(w))

# --- Step 4b: Compute weighted std from ALL like games (before zero-sales filter) ---
# This ensures bands are meaningful even when only 1 like game survives the sales filter
_sw = F.sum(F.col(weight_col_name))
unfiltered_std = train_data.groupBy("test_game_name", "month_index").agg(
    *[
        F.when(_sw > 0,
               F.sqrt(F.greatest(
                   F.sum(F.col(c) * F.col(c) * F.col(weight_col_name)) / _sw -
                   F.pow(F.sum(F.col(c) * F.col(weight_col_name)) / _sw, 2),
                   F.lit(0.0)
               ))
        ).otherwise(F.lit(0.0)).alias(f"{c}_std")
        for c in metric_cols
    ]
)

# --- Step 5: Exclude like games with zero total sales (no signal) ---
total_sales = train_data.groupBy("test_game_name", "game_name").agg(
    F.sum(sales_cl).alias("total_sales")
)
valid_games = total_sales.filter(F.col("total_sales") > 0).select("test_game_name", "game_name")
train_data = train_data.join(valid_games, ["test_game_name", "game_name"], "inner")

# --- Step 6: Determine curve length per test game ---
game_lengths = train_data.groupBy("test_game_name", "game_name").agg(
    F.max("month_index").alias("game_length")
)
if curve_length_method == 'min':
    curve_lengths = game_lengths.groupBy("test_game_name").agg(F.min("game_length").alias("curve_length"))
elif curve_length_method == 'max':
    curve_lengths = game_lengths.groupBy("test_game_name").agg(F.max("game_length").alias("curve_length"))
else:
    curve_lengths = game_lengths.groupBy("test_game_name").agg(
        F.round(F.avg("game_length")).cast("int").alias("curve_length")
    )
curve_lengths = curve_lengths.withColumn(
    "curve_length",
    F.least(F.col("curve_length"), F.lit(max_curve_length))
)

# Trim to curve length
train_data = train_data.join(curve_lengths, "test_game_name", "inner")
train_data = train_data.filter(F.col("month_index") <= F.col("curve_length"))

# --- Step 7: Weighted aggregation across like games per ordinal month ---
sum_weight = F.sum(weight_col_name)
agg_exprs = [
    F.when(sum_weight > 0, F.sum(F.col(c) * F.col(weight_col_name)) / sum_weight)
     .otherwise(0.0).alias(c)
    for c in metric_cols
]
agg_exprs.append(sum_weight.alias(f"total_{weight_col_name}"))
agg_exprs.append(F.countDistinct("game_name").alias("n_like_games_used"))
agg_exprs.append(F.sort_array(F.collect_set("game_name")).alias("like_games_used"))

predicted_curve = train_data.groupBy("test_game_name", "month_index", "curve_length", "own_status").agg(*agg_exprs)
predicted_curve = predicted_curve.orderBy("test_game_name", "month_index")
predicted_curve = predicted_curve.withColumn("run_id", F.lit(run_id))

# --- Step 8: Cumulative sum of sales over the curve ---
w_cum = Window.partitionBy("test_game_name").orderBy("month_index")
predicted_curve = predicted_curve.withColumn(f"cumsum_{sales_cl}", F.sum(sales_cl).over(w_cum))

# --- Step 9: Join unfiltered std (from ALL like games) and compute ±2σ bands ---
predicted_curve = predicted_curve.join(unfiltered_std, ["test_game_name", "month_index"], "left")
for c in metric_cols:
    predicted_curve = predicted_curve.withColumn(f"{c}_upper", F.col(c) + 2 * F.col(f"{c}_std"))
    predicted_curve = predicted_curve.withColumn(f"{c}_lower", F.col(c) - 2 * F.col(f"{c}_std"))

# Cumulative sales bands (cumulate upper/lower separately)
predicted_curve = predicted_curve.withColumn(f"cumsum_{sales_cl}_upper", F.sum(F.col(f"{sales_cl}_upper")).over(w_cum))
predicted_curve = predicted_curve.withColumn(f"cumsum_{sales_cl}_lower", F.sum(F.col(f"{sales_cl}_lower")).over(w_cum))

print(f"Curve length method: {curve_length_method} | Max curve length: {max_curve_length}")
display(predicted_curve)

In [0]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Get list of test games and create interactive widget ---
test_games = [row.test_game_name for row in predicted_curve.select("test_game_name").distinct().orderBy("test_game_name").collect()]
dbutils.widgets.dropdown("test_game", test_games[0], test_games, "Select Test Game")
selected_game = dbutils.widgets.get("test_game")

# --- Predicted curve for selected game ---
pred_pd = predicted_curve.filter(F.col("test_game_name") == selected_game).orderBy("month_index").toPandas()
curve_len = int(pred_pd['curve_length'].iloc[0]) if len(pred_pd) > 0 else max_curve_length

# --- Actual data for selected game (if it exists in time series) ---
actual_ts = time_series_df.filter(F.col("game_name") == selected_game)
w = Window.partitionBy("game_name").orderBy("yearmonth")
actual_ts = actual_ts.withColumn("month_index", F.row_number().over(w))

# Join actual performance with actual sales
actual_sales = sales_df.dropDuplicates(["beginning_month_date", "revenue_type", sales_cl, "ep_theme_name"]) \
    .filter(F.col("ep_theme_name") == selected_game) \
    .groupBy("ep_theme_name", "beginning_month_date").agg(F.sum(F.col(sales_cl).cast("double")).alias(sales_cl))

actual_with_sales = actual_ts.join(
    actual_sales,
    (actual_ts.game_name == actual_sales.ep_theme_name) & (actual_ts.yearmonth == actual_sales.beginning_month_date),
    "left"
).select(
    actual_ts.game_name,
    "month_index",
    F.col("no_of_slots").cast("double").alias("no_of_slots"),
    *[F.col(c).cast("double").alias(c) for c in performance],
    F.coalesce(F.col(sales_cl), F.lit(0.0)).alias(sales_cl)
).filter(F.col("month_index") <= curve_len).orderBy("month_index")

actual_pd = actual_with_sales.toPandas()
has_actual = len(actual_pd) > 0

# --- Compute cumulative sum for actual data ---
if has_actual:
    actual_pd[sales_cl_cumsum] = actual_pd[sales_cl].cumsum()

# --- Plot: 4 subplots (coin_in, theo_win, sales, cumulative sales) ---
fig, axes = plt.subplots(4, 1, figsize=(12, 13), sharex=True)
fig.suptitle(f"Actual vs Predicted: {selected_game}", fontsize=14, fontweight='bold')

metrics = [
    ('avg_coin_in_index_vs_house', 'Avg Coin-In Index vs House'),
    ('avg_theo_net_win_index_vs_house', 'Avg Theo Net Win Index vs House'),
    (sales_cl, 'Sales (units)'),
    (sales_cl_cumsum, 'Cumulative Sales (units)')
]

pred_cols = pred_pd.columns.tolist()
actual_cols = actual_pd.columns.tolist() if has_actual else []

for ax, (col, label) in zip(axes, metrics):
    ax.plot(pred_pd['month_index'], pred_pd[col], 'b-o', markersize=5, label='Predicted (like games)')
    # ±2σ confidence band
    upper_col, lower_col = f'{col}_upper', f'{col}_lower'
    if upper_col in pred_cols and lower_col in pred_cols:
        ax.fill_between(pred_pd['month_index'], pred_pd[lower_col], pred_pd[upper_col],
                        alpha=0.18, color='blue', label='±2σ band')
    if has_actual and col in actual_cols:
        ax.plot(actual_pd['month_index'], actual_pd[col], 'r--s', markersize=5, label='Actual')
    ax.set_ylabel(label)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[3].set_xlabel('Month Index')

if not has_actual:
    fig.text(0.5, 0.02, '⚠ No actual data found for this test game in the time series', ha='center', fontsize=11, color='orange')

plt.tight_layout()
plt.show()

print(f"Game: {selected_game} | Curve length: {curve_len} | Actual data available: {has_actual}")
if has_actual:
    print(f"Like games used: {pred_pd['like_games_used'].iloc[0]}")

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

# --- Collect predicted curves (cumulative sales) for all test games ---
curve_pd = predicted_curve.select(
    "test_game_name", "month_index", sales_cl_cumsum,
    f"{sales_cl_cumsum}_upper", f"{sales_cl_cumsum}_lower"
).orderBy("test_game_name", "month_index").toPandas()

# --- Define candidate models (no intercept) ---
def fit_linear(x, y):
    """y = a*x"""
    a = np.dot(x, y) / np.dot(x, x)
    return np.array([a]), lambda t: a * t

def fit_poly2(x, y):
    """y = a*x + b*x^2 (no intercept)"""
    X = np.column_stack([x, x**2])
    coeffs, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    return coeffs, lambda t: coeffs[0]*t + coeffs[1]*t**2

def fit_log(x, y):
    """y = a*log(x+1)"""
    log_x = np.log(x + 1)
    a = np.dot(log_x, y) / np.dot(log_x, log_x)
    return np.array([a]), lambda t: a * np.log(t + 1)

def r_squared(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot == 0:
        return 0.0
    return 1.0 - ss_res / ss_tot

def ensure_nonneg(y_pred):
    """Shift curve up if any value is negative."""
    min_val = np.min(y_pred)
    if min_val < 0:
        return y_pred - min_val
    return y_pred

# --- Fit all 3 models per test game and keep best ---
results = []
model_info = []

for game, gdf in curve_pd.groupby("test_game_name"):
    x = gdf["month_index"].values.astype(float)
    y = gdf[sales_cl_cumsum].values.astype(float)
    upper = gdf[f"{sales_cl_cumsum}_upper"].values.astype(float)
    lower = gdf[f"{sales_cl_cumsum}_lower"].values.astype(float)

    candidates = [
        ("Linear", fit_linear),
        ("Polynomial_deg2", fit_poly2),
        ("Logarithmic", fit_log),
    ]

    best_r2 = -np.inf
    best_name = None
    best_pred = None
    best_func = None

    all_fits = {}
    for name, fit_fn in candidates:
        coeffs, func = fit_fn(x, y)
        y_hat = ensure_nonneg(func(x))
        score = r_squared(y, y_hat)
        all_fits[name] = {"r2": score, "y_hat": y_hat, "func": func}
        if score > best_r2:
            best_r2 = score
            best_name = name
            best_pred = y_hat
            best_func = func

    # Store model selection info
    model_info.append({
        "test_game_name": game,
        "best_model": best_name,
        "R2": round(best_r2, 4),
        "R2_linear": round(all_fits["Linear"]["r2"], 4),
        "R2_poly2": round(all_fits["Polynomial_deg2"]["r2"], 4),
        "R2_log": round(all_fits["Logarithmic"]["r2"], 4),
    })

    # Build forecast output
    for i, mi in enumerate(x.astype(int)):
        results.append({
            "test_game_name": game,
            "release_month": mi,
            "ForecastUnitBaseline": best_pred[i],
            "ForecastUnitBaseline_upper": ensure_nonneg(np.array([best_func(x)[j] + (upper[j] - y[j]) for j in range(len(x))]))[i] if i < len(upper) else None,
            "ForecastUnitBaseline_lower": ensure_nonneg(np.array([best_func(x)[j] + (lower[j] - y[j]) for j in range(len(x))]))[i] if i < len(lower) else None,
            "best_model": best_name,
            "R2": round(best_r2, 4),
        })

forecast_df = pd.DataFrame(results)
model_summary = pd.DataFrame(model_info)

print("=== Model Selection Summary ===")
display(spark.createDataFrame(model_summary))
print(f"\n=== Forecast Unit Baseline (all games) ===")
display(spark.createDataFrame(forecast_df))

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Configuration ---
dbutils.widgets.dropdown(
    "shift_flag",
    "True",
    ["True", "False"],
    "Shift performance alignment with sales"
)

shift_flag = (
    dbutils.widgets.get("shift_flag")
    .strip()
    .lower()
    == "true"
)

# ============================================================
# Pull actual data for test games
# ============================================================
test_game_list = forecast_df['test_game_name'].unique().tolist()

# Actual performance from time_series_df
actual_perf_spark = time_series_df.filter(F.col("game_name").isin(test_game_list))
w_perf = Window.partitionBy("game_name").orderBy("yearmonth")
actual_perf_spark = actual_perf_spark.withColumn("release_month", F.row_number().over(w_perf))
actual_perf_pd = actual_perf_spark.select(
    F.col("game_name").alias("test_game_name"),
    "release_month",
    *[F.col(c).cast("double").alias(f"actual_{c}") for c in performance]
).toPandas()

# Actual sales from sales_df
actual_sales_spark = (
    sales_df
    .dropDuplicates(["beginning_month_date", "revenue_type", sales_cl, "ep_theme_name"])
    .filter(F.col("ep_theme_name").isin(test_game_list))
    .groupBy("ep_theme_name", "beginning_month_date")
    .agg(F.sum(F.col(sales_cl).cast("double")).alias(f"actual_{sales_cl}"))
)
w_s = Window.partitionBy("ep_theme_name").orderBy("beginning_month_date")
actual_sales_spark = actual_sales_spark.withColumn("release_month", F.row_number().over(w_s))
actual_sales_pd = actual_sales_spark.select(
    F.col("ep_theme_name").alias("test_game_name"),
    "release_month",
    f"actual_{sales_cl}"
).toPandas()

# Merge actual perf + sales
actuals_pd = actual_perf_pd.merge(actual_sales_pd, on=["test_game_name", "release_month"], how="outer")
actuals_pd = actuals_pd.sort_values(["test_game_name", "release_month"]).reset_index(drop=True)

# ============================================================
# Build forecast_enriched: forecast_df + run_id + shifted actuals
# (shift applied to ACTUALS per game based on release_month)
# ============================================================
forecast_enriched = forecast_df.copy()
forecast_enriched['run_id'] = run_id

# Merge actuals into forecast
forecast_enriched = forecast_enriched.merge(actuals_pd, on=["test_game_name", "release_month"], how="left")

# Join predicted performance from predicted_curve
pred_perf_pd = predicted_curve.select(
    "test_game_name",
    F.col("month_index").alias("release_month"),
    *[F.col(c).alias(f"predicted_{c}") for c in performance]
).toPandas()
forecast_enriched = forecast_enriched.merge(pred_perf_pd, on=["test_game_name", "release_month"], how="left")

# --- Per-game shift: align first non-zero actual sale to first non-zero forecast ---
shifted_games = []
for game, gdf in forecast_enriched.groupby("test_game_name"):
    gdf = gdf.sort_values("release_month").reset_index(drop=True)
    actual_sale_col = f"actual_{sales_cl}"

    # First month where extrapolated forecast > 0
    fc_nonzero = gdf[gdf["ForecastUnitBaseline"] > 0]
    first_fc_month = int(fc_nonzero["release_month"].iloc[0]) if len(fc_nonzero) > 0 else 1

    # First month where actual sales > 0
    has_actuals = actual_sale_col in gdf.columns and gdf[actual_sale_col].notna().any()
    shift = 0
    if has_actuals:
        act_nonzero = gdf[(gdf[actual_sale_col].notna()) & (gdf[actual_sale_col] > 0)]
        if len(act_nonzero) > 0:
            first_act_month = int(act_nonzero["release_month"].iloc[0])
            shift = first_fc_month - first_act_month

    gdf["shift_months"] = shift

    # Build shifted actuals (shift actuals to align with forecast timeline)
    game_actuals_full = actuals_pd[actuals_pd["test_game_name"] == game].copy()
    if len(game_actuals_full) > 0 and shift != 0:
        game_actuals_full["release_month"] = game_actuals_full["release_month"] + shift
        # Shifted sales
        shifted_sales = game_actuals_full[["test_game_name", "release_month", actual_sale_col]].rename(
            columns={actual_sale_col: f"shifted_actual_{sales_cl}"}
        )
        gdf = gdf.merge(shifted_sales, on=["test_game_name", "release_month"], how="left")
        # Shifted performance (only if shift_flag)
        if shift_flag:
            perf_shift_cols = {f"actual_{c}": f"shifted_actual_{c}" for c in performance}
            shifted_perf = game_actuals_full[["test_game_name", "release_month"] + [f"actual_{c}" for c in performance]].rename(columns=perf_shift_cols)
            gdf = gdf.merge(shifted_perf, on=["test_game_name", "release_month"], how="left")
        else:
            for c in performance:
                gdf[f"shifted_actual_{c}"] = gdf.get(f"actual_{c}")
    else:
        # No shift needed — copy as-is
        gdf[f"shifted_actual_{sales_cl}"] = gdf.get(actual_sale_col)
        for c in performance:
            gdf[f"shifted_actual_{c}"] = gdf.get(f"actual_{c}")

    shifted_games.append(gdf)

forecast_final_df = pd.concat(shifted_games, ignore_index=True)

# Cumulative shifted actual sales
forecast_final_df[f"shifted_actual_{sales_cl_cumsum}"] = (
    forecast_final_df.groupby("test_game_name")[f"shifted_actual_{sales_cl}"].cumsum()
)

# ============================================================
# Compute per-game error metrics and place in model_summary
# ============================================================
error_records = []
for game, gdf in forecast_final_df.groupby("test_game_name"):
    rec = {"test_game_name": game}

    # --- Cumulative sales error (MAPE & RMSE) ---
    valid_sales = gdf[
        gdf[f"shifted_actual_{sales_cl_cumsum}"].notna() &
        (gdf[f"shifted_actual_{sales_cl_cumsum}"] != 0)
    ]
    if len(valid_sales) > 0:
        errors_s = valid_sales["ForecastUnitBaseline"] - valid_sales[f"shifted_actual_{sales_cl_cumsum}"]
        rec["MAPE_cum_sales"] = round(
            (errors_s.abs() / valid_sales[f"shifted_actual_{sales_cl_cumsum}"].abs()).mean() * 100, 2
        )
        rec["RMSE_cum_sales"] = round(np.sqrt((errors_s ** 2).mean()), 4)
        rec["MAE_cum_sales"] = round(errors_s.abs().mean(), 4)
    else:
        rec["MAPE_cum_sales"] = np.nan
        rec["RMSE_cum_sales"] = np.nan
        rec["MAE_cum_sales"] = np.nan

    # --- Performance errors (MAPE per metric) ---
    for c in performance:
        valid_perf = gdf[
            gdf[f"shifted_actual_{c}"].notna() &
            (gdf[f"shifted_actual_{c}"] != 0)
        ]
        if len(valid_perf) > 0:
            errors_p = valid_perf[f"predicted_{c}"] - valid_perf[f"shifted_actual_{c}"]
            rec[f"MAPE_{c}"] = round(
                (errors_p.abs() / valid_perf[f"shifted_actual_{c}"].abs()).mean() * 100, 2
            )
            rec[f"RMSE_{c}"] = round(np.sqrt((errors_p ** 2).mean()), 4)
        else:
            rec[f"MAPE_{c}"] = np.nan
            rec[f"RMSE_{c}"] = np.nan

    # Shift applied
    rec["shift_months"] = int(gdf["shift_months"].iloc[0])
    error_records.append(rec)

error_df = pd.DataFrame(error_records)

# ============================================================
# DataFrame 1: model_summary + run_id + error metrics
# ============================================================
model_summary_with_id = model_summary.copy()
model_summary_with_id['run_id'] = run_id
model_summary_with_id = model_summary_with_id.merge(error_df, on="test_game_name", how="left")

# ============================================================
# DataFrame 2: forecast_df + run_id + shifted actuals (no error cols)
# ============================================================
forecast_display_cols = [
    "test_game_name", "release_month", "run_id", "best_model", "R2",
    "ForecastUnitBaseline", "ForecastUnitBaseline_upper", "ForecastUnitBaseline_lower",
    f"shifted_actual_{sales_cl}", f"shifted_actual_{sales_cl_cumsum}", "shift_months",
]
for c in performance:
    forecast_display_cols += [f"predicted_{c}", f"shifted_actual_{c}"]
forecast_display_cols = [c for c in forecast_display_cols if c in forecast_final_df.columns]
forecast_with_actuals = forecast_final_df[forecast_display_cols]

# ============================================================
# Display
# ============================================================
print("=== DataFrame 1: Model Summary + Run ID + Errors ===")
display(spark.createDataFrame(model_summary_with_id))

print(f"\n=== DataFrame 2: Forecast + Shifted Actuals (shift_flag={shift_flag}) ===")
print(f"Games with actual data: {forecast_final_df[forecast_final_df[f'shifted_actual_{sales_cl}'].notna()]['test_game_name'].nunique()} / {forecast_final_df['test_game_name'].nunique()}")
display(spark.createDataFrame(forecast_with_actuals))

In [0]:
import matplotlib.pyplot as plt
import numpy as np

selected_game = dbutils.widgets.get("test_game")

# --- Filter forecast for selected game ---
game_forecast = forecast_df[forecast_df["test_game_name"] == selected_game].sort_values("release_month")
game_curve = curve_pd[curve_pd["test_game_name"] == selected_game].sort_values("month_index")

if game_forecast.empty:
    print(f"No forecast data for '{selected_game}'")
else:
    x = game_forecast["release_month"].values
    y_raw = game_curve[sales_cl_cumsum].values
    y_fit = game_forecast["ForecastUnitBaseline"].values
    y_upper = game_forecast["ForecastUnitBaseline_upper"].values
    y_lower = game_forecast["ForecastUnitBaseline_lower"].values
    best_model = game_forecast["best_model"].iloc[0]
    r2 = game_forecast["R2"].iloc[0]

    # Also recompute all 3 candidate curves for visualization
    y_raw_arr = y_raw.astype(float)
    x_f = x.astype(float)

    # Linear
    a_lin = np.dot(x_f, y_raw_arr) / np.dot(x_f, x_f)
    y_linear = np.maximum(a_lin * x_f, 0)

    # Poly2
    X_poly = np.column_stack([x_f, x_f**2])
    coeffs_poly, _, _, _ = np.linalg.lstsq(X_poly, y_raw_arr, rcond=None)
    y_poly = coeffs_poly[0]*x_f + coeffs_poly[1]*x_f**2
    y_poly = y_poly - min(y_poly.min(), 0)

    # Log
    log_x = np.log(x_f + 1)
    a_log = np.dot(log_x, y_raw_arr) / np.dot(log_x, log_x)
    y_log = np.maximum(a_log * np.log(x_f + 1), 0)

    # --- Plot all 3 models + best selection ---
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"Forecast Model Fits: {selected_game}\nBest Model: {best_model} (R\u00b2={r2})",
                 fontsize=13, fontweight='bold')

    models_data = [
        ("Linear: y=a\u00b7x", y_linear, r_squared(y_raw_arr, y_linear)),
        ("Poly2: y=a\u00b7x+b\u00b7x\u00b2", y_poly, r_squared(y_raw_arr, y_poly)),
        ("Log: y=a\u00b7log(x+1)", y_log, r_squared(y_raw_arr, y_log)),
    ]

    for ax, (title, y_model, r2_val) in zip(axes, models_data):
        ax.scatter(x, y_raw_arr, color='black', s=40, zorder=5, label='Observed')
        ax.plot(x, y_model, linewidth=2, label=f'Fit (R\u00b2={r2_val:.4f})')
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Release Month')
        ax.set_ylabel('Cumulative Sales')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # --- Final forecast chart with confidence band ---
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    ax2.scatter(x, y_raw_arr, color='black', s=50, zorder=5, label='Observed (like-game avg)')
    ax2.plot(x, y_fit, 'b-o', markersize=6, linewidth=2, label=f'Forecast ({best_model}, R\u00b2={r2})')
    ax2.fill_between(x, y_lower, y_upper, alpha=0.18, color='blue', label='\u00b12\u03c3 band')
    ax2.set_title(f"Official Forecast Unit Baseline: {selected_game}", fontsize=13, fontweight='bold')
    ax2.set_xlabel('Release Month')
    ax2.set_ylabel('Forecast Unit Baseline (cumulative units)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"\nSelected model: {best_model} | R\u00b2 = {r2}")
    print(f"Forecast curve length: {len(x)} months")